# Relatório de Análise e Treinamento - Gerado por Shaula

**Dataset:** Análise do dataset fornecido.
**Objetivo:** Prever a coluna `Is_Binary_Indicator_Of_Cancer_Diagnosis`.

In [ ]:
# Célula de Configuração
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

# Simulação do carregamento de dados para o notebook ser autocontido
# df = pd.read_csv('caminho/para/seus/dados.csv')

## Passo 1: Avaliação Inicial do Dataset

In [ ]:
df.info()
display(df.describe(include='all').round(2))

### Análise da Shaula (Passo 1)

1. **Tipo de Problema:**
   - Esta é uma tarefa de **Classificação**. A justificativa baseia-se na natureza da coluna-alvo 'Is_Binary_Indicator_Of_Cancer_Diagnosis', que é do tipo booleano (bool) e possui apenas dois valores únicos (True/False). Isso sugere que o objetivo é classificar as instâncias como positivas ou negativas para o diagnóstico de câncer.

2. **Qualidade dos Dados (Primeira Impressão):**
   - **Dados Ausentes:** 
     - Com base nas informações fornecidas, não há colunas com valores nulos, pois todas as colunas têm contagens de entradas não nulas iguais ao total de entradas (39998). No entanto, a presença de categorias como 'Missing' na coluna 'Cancer_Type STRING' pode indicar dados ausentes codificados como strings.
   - **Tipos de Dados:** 
     - Várias colunas categóricas são armazenadas como objetos do tipo string ('Radiologists_Assessment STRING', 'Comparison_Mammogram_From_Mammography STRING', etc.). Será necessário converter esses dados para um formato adequado para modelagem, como categorias numéricas usando codificação de rótulos ou codificação one-hot.
   - **Escalas Numéricas:**
     - A coluna 'Patients_Study_ID INTEGER' parece um identificador único e não deve ser usada diretamente na modelagem. A coluna 'Age_At_The_Time_Of_Mammography INTEGER' é a única coluna numérica relevante para análise, e sua escala parece razoável (mínimo de 60 e máximo de 89).
   - **Cardinalidade Categórica:**
     - A coluna 'Cancer_Type STRING' tem uma alta cardinalidade, com 1897 categorias únicas, o que pode ser um desafio em termos de codificação e modelagem. Outras colunas categóricas parecem ter um número manejável de categorias únicas (entre 2 e 6).

3. **Hipótese Inicial:**
   - A fase de **limpeza de dados** pode ser a mais desafiadora, especialmente devido à presença de valores ausentes codificados como strings ('Missing') e a necessidade de converter várias colunas categóricas para um formato adequado para modelagem. Além disso, lidar com a alta cardinalidade da coluna 'Cancer_Type STRING' pode exigir técnicas específicas de feature engineering.

4. **Sugestão de Próximo Passo:**
   - O próximo passo lógico é realizar uma **Análise Exploratória de Dados (AED)** mais aprofundada. Isso incluirá visualizar as distribuições das variáveis, verificar correlações entre as características e a variável-alvo, identificar possíveis outliers e confirmar a presença de valores ausentes codificados como strings. A AED fornecerá insights valiosos sobre a estrutura e as características dos dados, fundamental para guiar o pré-processamento e a modelagem subsequente.

## Passo 2: Análise Exploratória de Dados (AED)

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='Is_Binary_Indicator_Of_Cancer_Diagnosis', data=df, palette='viridis', order=df['Is_Binary_Indicator_Of_Cancer_Diagnosis'].value_counts().index)
plt.title('Distribuição da Variável-Alvo (Is_Binary_Indicator_Of_Cancer_Diagnosis)')
plt.savefig('exploracao/grafico_distribuicao_alvo.png')
plt.show()

### Análise da Shaula (Passo 2)

Para realizar uma Análise Exploratória de Dados (AED) eficaz no contexto de classificação com a variável-alvo 'Is_Binary_Indicator_Of_Cancer_Diagnosis', podemos seguir o seguinte plano de ação:

### 1. Análise da Variável-Alvo

**Pergunta Principal:** Como está distribuída a variável-alvo 'Is_Binary_Indicator_Of_Cancer_Diagnosis'? Existe desbalanceamento de classes?

**Ação:**
- **Gráfico a Usar:** Um gráfico de barras (bar plot) para visualizar a distribuição das classes. Se a variável for binária, um gráfico de barras simples mostrará o número de ocorrências de cada classe (0 e 1).
- **Problemas a Procurar:** Desbalanceamento de classes, onde uma classe é significativamente mais frequente que a outra, o que pode afetar o desempenho dos modelos de classificação.

### 2. Análise de Preditores Numéricos

**Pergunta Principal:** Qual é a relação entre as features numéricas e a variável-alvo?

**Ação:**
- **Gráfico a Usar:** Box plots e histogramas separados por classe da variável-alvo para cada feature numérica. Isso ajuda a visualizar a distribuição das features numéricas em relação a cada classe da variável-alvo.
- **Ferramenta Estatística Principal:** Análise de correlação e testes de hipótese como o teste t de Student ou o teste de Mann-Whitney, dependendo da distribuição dos dados, para verificar diferenças estatisticamente significativas entre as classes.

### 3. Análise de Preditores Categóricos

**Pergunta Principal:** Como as features categóricas se relacionam com a variável-alvo?

**Ação:**
- **Gráfico a Usar:** Gráficos de barras ou gráficos de mosaico (mosaic plots) para cada feature categórica, segmentados por classes da variável-alvo, para observar a distribuição relativa das categorias.
- **Ferramenta Estatística:** Teste qui-quadrado para independência para verificar se há uma associação estatisticamente significativa entre as features categóricas e a variável-alvo.

### 4. Conclusão e Próximo Passo

**Insight Esperado:** O insight mais importante da AED é entender quais preditores têm uma relação significativa com a variável-alvo. Isso inclui identificar potenciais problemas de desbalanceamento de classe, detectar outliers ou valores atípicos em features numéricas, e compreender as associações entre features categóricas e a variável-alvo. A AED também ajudará a identificar quais variáveis podem ser mais informativas para modelos preditivos subsequentes.

**Próximo Passo:** Com base nos insights obtidos, o próximo passo seria preparar os dados para modelagem, o que pode incluir técnicas de balanceamento de classes, transformação ou normalização de features, e seleção ou engenharia de features para melhorar o desempenho do modelo.

## Passo 3: Estratégia de Pré-processamento e Pipeline

### Análise da Shaula (Passo 3)

Para construir um pipeline de pré-processamento robusto com o Scikit-Learn, vamos abordar cada aspecto mencionado no contexto e projetar um `ColumnTransformer` que trate adequadamente as características dos dados descritos. Vamos detalhar cada passo necessário:

### 1. Estratégia de Divisão de Dados
Para lidar com o desbalanceamento da variável-alvo ao dividir os dados em conjuntos de treino e teste, utilizaremos o parâmetro `stratify` na função `train_test_split`. Isso garantirá que a proporção das classes na variável-alvo seja mantida em ambos os conjuntos de dados.

```python
from sklearn.model_selection import train_test_split

# Supondo que X seja o conjunto de features e y a variável alvo
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
```

### 2. Pipeline para Features Numéricas
Para processar as features numéricas, considerando a distribuição assimétrica, o pipeline incluirá:

- **Imputação de Valores Ausentes:** Usaremos `SimpleImputer` para preencher valores ausentes com a mediana, que é robusta a outliers.
- **Transformação Logarítmica:** Aplicaremos uma transformação logarítmica para lidar com a assimetria das distribuições.
- **Escalonamento:** Utilizaremos `StandardScaler` para escalonar os dados, ajustando-os a uma média de 0 e desvio padrão de 1.

```python
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('log_transform', FunctionTransformer(func=np.log1p, validate=True)),
    ('scaler', StandardScaler())
])
```

### 3. Pipeline para Features Categóricas
Para as features categóricas, o pipeline incluirá:

- **Imputação de Valores Ausentes:** Usaremos `SimpleImputer` para preencher valores ausentes com a moda (valor mais frequente).
- **Codificação One-Hot:** Utilizaremos `OneHotEncoder` para transformar as categorias em uma representação binária.

```python
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
```

### 4. Tratamentos Especiais
Dado que as features 'A' e 'B' têm distribuições assimétricas, já incorporamos a transformação logarítmica no pipeline numérico. Caso existam outras transformações específicas para colunas adicionais, podemos defini-las como funções personalizadas e adicioná-las ao pipeline correspondente.

### Integração no ColumnTransformer
Finalmente, integramos os pipelines numérico e categórico em um `ColumnTransformer`, especificando quais colunas devem passar por cada pipeline.

```python
from sklearn.compose import ColumnTransformer

# Supondo que 'num_features' e 'cat_features' sejam listas com os nomes das colunas numéricas e categóricas
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features)
    ])
```

Esse `ColumnTransformer` pode então ser integrado em um pipeline completo para treinamento de modelos, garantindo que todas as etapas de pré-processamento sejam aplicadas de forma consistente e automatizada.

## Passo 4: Treinamento do Modelo de Baseline e Avaliação